In [ ]:
import ROOT
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# load library from the Mach3 build
ROOT.gSystem.AddDynamicPath("../build/")
ROOT.gSystem.Load("libduneanaobj_StandardRecord")

In [ ]:
# open CAF .root, and get event tree
f = ROOT.TFile("/storage/1/st15719/dune/atmo/Mach3_Inputs/CAFs/caf_new_sum.2.6M_weighted.root")
caf_tree = f.Get("cafTree")

In [ ]:
var_labels = ["true_pdg", "true_energy", "true_n_prim_mu", "reco_ok", "reco_n_prim", "reco_energy_lep_calo", "reco_energy_e_calo", "reco_energy_calo"]
data_arrays=[]

n_processed_events=0
for entry in caf_tree:

    data_array = {key: None for key in var_labels}

    # get truth information
    n_true_nu = entry.rec.mc.nu.size()
    if n_true_nu != 1:
        print(f"ERROR: unexpected mc truth size: {n_true_nu}; should be 1")
        break
    true_nu = entry.rec.mc.nu[0]
    
    data_array["true_energy"] = true_nu.E
    # true nu species
    data_array["true_pdg"] = true_nu.pdg

     # n primary true muons
    true_n_primary_mu=0

    for nu_primary in true_nu.prim:
        primary_pdg = nu_primary.pdg
        if abs(primary_pdg) == 13:
            true_n_primary_mu=true_n_primary_mu+1

    data_array["true_n_prim_mu"] = true_n_primary_mu
    ##

    # get reco information
    reco_ok=False
    reco_n_prim = np.nan
    reco_energy_lep_calo = np.nan
    reco_energy_e_calo = np.nan
    reco_energy_calo = np.nan

    n_reco_nu = entry.rec.common.ixn.pandora.size()

    if n_reco_nu == 0:
        print(f"WARNING: no reco nu!")
    elif n_reco_nu != 1:
        print(f"ERROR: unexpected n of reco nu: {n_reco_nu}")
        break
    else:
        reco_ok=True

    if reco_ok:
        reco_nu = entry.rec.common.ixn.pandora[0]
        reco_n_prim=0
        reco_energy_lep_calo = reco_nu.Enu.lep_calo
        reco_energy_e_calo = reco_nu.Enu.e_calo
        reco_energy_calo = reco_nu.Enu.calo
        
        for reco_pfp in reco_nu.part.pandora:
            if reco_pfp.primary:
                reco_n_prim=reco_n_prim+1

    data_array["reco_ok"] = reco_ok
    data_array["reco_n_prim"] = reco_n_prim
    data_array["reco_energy_lep_calo"] = reco_energy_lep_calo
    data_array["reco_energy_e_calo"] = reco_energy_e_calo
    data_array["reco_energy_calo"] = reco_energy_calo

    data_arrays.append(list(data_array.values()))
    ###

    n_processed_events=n_processed_events+1

print(f"Processed {n_processed_events} events")

In [ ]:
df = pd.DataFrame(data_arrays, columns=var_labels)
print(df)

In [ ]:
# filter out events with no reco info
df_reco_ok=df[df["reco_ok"] == True]
print(df_reco_ok)

# column to array
true_nu_e = df_reco_ok['true_energy'].to_numpy()
print(true_nu_e)

# make plot
plt.hist(true_nu_e) 
plt.xlabel("true nu E [GeV]") 
plt.show()